# Differential Equations — Session 20
## Section 4.8: Green’s Functions

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Interpret a Green’s function as an impulse-response kernel.
2. derive the zero-initial-data integral from variation of parameters.
3. compute convolution-type responses.
4. compare impulse, step, pulse, and sinusoidal forcing.
5. verify Green-function solutions numerically.
6. construct a simple boundary-value Green’s function.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–18 min | Rest response and kernel idea |\n| 18–42 min | IVP Green’s function derivation |\n| 42–64 min | Convolution and forcing experiments |\n| 64–78 min | Numerical verification |\n| 78–88 min | BVP Green’s function |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Theorem 4.8-A — Zero-state IVP representation

For

$$y''+P(x)y'+Q(x)y=f(x),\qquad y(x_0)=y'(x_0)=0,$$

let $y_1,y_2$ be a fundamental set with Wronskian $W(t)$. Then

$$y(x)=\int_{x_0}^xG(x,t)f(t)\,dt,$$

where the **Green’s function** is

$$G(x,t)=\frac{y_1(t)y_2(x)-y_2(t)y_1(x)}{W(t)}.$$

The kernel is causal: only $t\le x$ contributes to the IVP response.

For nonzero initial data, add the homogeneous solution satisfying the initial conditions.

### Definition 4.8-B — Boundary-value Green’s function

For a BVP, the Green’s function is piecewise and is constructed to satisfy the boundary conditions in $x$, continuity at $x=t$, and a unit derivative jump.

### Classroom Checkpoint — Green’s Function Interpretation

For a zero-initial-data linear problem, how is the response to forcing $f$ expressed using a Green’s function $G$?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Oscillator kernel

For $y''+\omega^2y=f$, zero initial data give

$$G(x,t)=\frac{\sin[\omega(x-t)]}{\omega},\qquad t\le x.$$

In [ ]:
omega=2; xs=np.linspace(0,8,350); ts=np.linspace(0,8,350); X,T=np.meshgrid(xs,ts); G=np.where(T<=X,np.sin(omega*(X-T))/omega,np.nan)
plt.imshow(G,origin='lower',extent=[0,8,0,8],aspect='auto'); plt.colorbar(label='G(x,t)'); plt.xlabel('x'); plt.ylabel('t'); plt.title('Causal Green kernel'); plt.show()

## 2. Response to different inputs

In [ ]:
def green_response(kind='pulse',omega=2.0,Tmax=20):
    if kind=='step': f=lambda t:1.0
    elif kind=='pulse': f=lambda t:1.0 if 2<=t<=4 else 0.0
    elif kind=='sine': f=lambda t:np.sin(1.5*t)
    else: f=lambda t:np.exp(-(t-4)**2)
    x=np.linspace(0,Tmax,600); y=np.array([quad(lambda s:np.sin(omega*(xx-s))/omega*f(s),0,xx)[0] for xx in x]); forcing=np.array([f(xx) for xx in x])
    plt.plot(x,y,label='response'); plt.plot(x,forcing,linestyle='--',label='forcing'); plt.legend(); plt.show()
if WIDGETS_AVAILABLE: interact(green_response,kind=Dropdown(options=['step','pulse','sine','gaussian'],value='pulse'),omega=FloatSlider(min=.5,max=4,step=.1,value=2),Tmax=IntSlider(min=8,max=40,step=4,value=20))
else: green_response()

## 3. Verify convolution against an ODE solver

In [ ]:
omega=2.; f=lambda t:np.exp(-(t-3)**2); x=np.linspace(0,12,500)
yg=np.array([quad(lambda s:np.sin(omega*(xx-s))/omega*f(s),0,xx)[0] for xx in x])
sol=solve_ivp(lambda t,z:[z[1],f(t)-omega**2*z[0]],(0,12),[0,0],t_eval=x,rtol=1e-9,atol=1e-11)
plt.plot(x,yg,label='Green integral'); plt.plot(x,sol.y[0],linestyle='--',label='solve_ivp'); plt.legend(); plt.show(); print(np.max(np.abs(yg-sol.y[0])))

## 4. Boundary-value Green’s function

For

$$-y''=f(x),\qquad y(0)=y(1)=0,$$

one Green’s function is

$$G(x,t)=\begin{cases}x(1-t),&x\le t,\\t(1-x),&t\le x.\end{cases}$$

Then $y(x)=\int_0^1G(x,t)f(t)dt$.

In [ ]:
x=np.linspace(0,1,300); t=np.linspace(0,1,300); X,T=np.meshgrid(x,t); G=np.where(X<=T,X*(1-T),T*(1-X))
plt.imshow(G,origin='lower',extent=[0,1,0,1],aspect='equal'); plt.colorbar(); plt.xlabel('x'); plt.ylabel('t'); plt.title('Dirichlet BVP Green function'); plt.show()
# f=1 gives y=x(1-x)/2
yn=np.array([quad(lambda s:(xx*(1-s) if xx<=s else s*(1-xx)),0,1)[0] for xx in x]); plt.plot(x,yn,label='integral'); plt.plot(x,x*(1-x)/2,linestyle='--',label='exact'); plt.legend(); plt.show()

## Classroom Checkpoint — Exit Check

Why is the IVP kernel zero for $t>x$?

> Pause here. Let students commit to an answer before running the next cell.